# Spatial block-permutation MMD lab
This exercise permutes whole equal-sized blocks. The inferential unit is the block, not the individual point or pixel.

In [ ]:
import sys, numpy as np
sys.path.insert(0, '../implementations/python/distribution-shift')
from block_permutation import block_mmd_permutation_test

In [ ]:
rng=np.random.default_rng(21)
source_blocks=rng.normal(0,.5,size=(10,20,2))
target_blocks=rng.normal(.7,.5,size=(10,20,2))
result=block_mmd_permutation_test(source_blocks,target_blocks,bandwidth=1,n_permutations=499,random_state=8)
print('block-level MMD:',result['statistic'])
print('block permutation p-value:',result['p_value'])
print('permutation unit:',result['permutation_unit'])

## Design requirements
This test is only defensible when blocks are mutually independent and exchangeable under the null, have equal row counts and comparable sampling/measurement designs. Choose block geometry and size from the spatial dependence range and target estimand before inspecting results. The procedure does not repair dependence across blocks, unequal sampling designs, confounding or non-stationarity. It is not a generic cure for spatial autocorrelation.

## Variable-sized block example
The second API uses a block-level kernel and permits different observation counts per block. It still permutes whole blocks, not individual rows.

In [ ]:
from block_permutation import spatial_block_mmd_test
features=[]; labels=[]; block_ids=[]
for block in range(12):
    group=0 if block<6 else 1
    count=3+(block%3)
    features.append(rng.normal(1.5*group,.25,size=(count,2)))
    labels.extend([group]*count); block_ids.extend([block]*count)
variable_result=spatial_block_mmd_test(np.vstack(features),np.asarray(labels),np.asarray(block_ids),permutations=499,seed=8)
print('block-kernel MMD:',variable_result['mmd2_unbiased'])
print('block permutation p-value:',variable_result['p_value'])
print('block counts:',variable_result['n_blocks_group_0'],variable_result['n_blocks_group_1'])

The two designs have different statistics and data contracts. The equal-sized design flattens the assigned blocks and evaluates the observation-level MMD; the variable-sized design forms a kernel between blocks and evaluates an unbiased block-level MMD. Both require a defensible exchangeability argument for whole blocks.